## Data Cleaning and Preprocessing

### Import and load Medical_Appointment_No_Shows Dataset

In [58]:
import pandas as pd
df = pd.read_csv(r'E:\archive\dataset.csv')
print("--- FIRST 5 ROWS ---")
df.head()

--- FIRST 5 ROWS ---


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


### Check data types and missing values

In [59]:
print("--- DATASET INFO ---")
df.info()

--- DATASET INFO ---
<class 'pandas.DataFrame'>
RangeIndex: 110527 entries, 0 to 110526
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   PatientId       110527 non-null  float64
 1   AppointmentID   110527 non-null  int64  
 2   Gender          110527 non-null  str    
 3   ScheduledDay    110527 non-null  str    
 4   AppointmentDay  110527 non-null  str    
 5   Age             110527 non-null  int64  
 6   Neighbourhood   110527 non-null  str    
 7   Scholarship     110527 non-null  int64  
 8   Hipertension    110527 non-null  int64  
 9   Diabetes        110527 non-null  int64  
 10  Alcoholism      110527 non-null  int64  
 11  Handcap         110527 non-null  int64  
 12  SMS_received    110527 non-null  int64  
 13  No-show         110527 non-null  str    
dtypes: float64(1), int64(8), str(5)
memory usage: 17.6 MB


##### No null value found

### Get statistical summary of numerical columns

In [60]:
print("--- STATISTICAL SUMMARY ---")
df.describe()

--- STATISTICAL SUMMARY ---


,PatientId,AppointmentID,Age,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received
count,1.105270e+05,1.105270e+05,110527.000000,110527.000000,110527.000000,110527.000000,110527.000000,110527.000000,110527.000000
mean,1.474963e+14,5.675305e+06,37.088874,0.098266,0.197246,0.071865,0.030400,0.022248,0.321026
std,2.560949e+14,7.129575e+04,23.110205,0.297675,0.397921,0.258265,0.171686,0.161543,0.466873
min,3.921784e+04,5.030230e+06,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4.172614e+12,5.640286e+06,18.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,3.173184e+13,5.680573e+06,37.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,9.439172e+13,5.725524e+06,55.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
max,9.999816e+14,5.790484e+06,115.000000,1.000000,1.000000,1.000000,1.000000,4.000000,1.000000


### Check Duplicates

In [61]:
df.duplicated().sum()

np.int64(0)

In [62]:
df["AppointmentID"].duplicated().sum()

np.int64(0)

##### No duplicates found

### Check categorical values

In [63]:
df["Gender"].unique()

<ArrowStringArray>
['F', 'M']
Length: 2, dtype: str

In [64]:
df["No-show"].unique()

<ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str

In [65]:
df["Neighbourhood"].nunique()

81

In [66]:
for col in ["Scholarship", "Hipertension", "Diabetes", 
            "Alcoholism", "Handcap", "SMS_received"]:
    print(col, df[col].unique())

Scholarship [0 1]
Hipertension [1 0]
Diabetes [0 1]
Alcoholism [0 1]
Handcap [0 1 2 3 4]
SMS_received [0 1]


In [67]:
df["Handcap"].value_counts().sort_index()

Handcap
0    108286
1      2042
2       183
3        13
4         3
Name: count, dtype: int64

Numbers could be level of handcap so we will leave it unchanged.

### Check invalid Age and remove

In [68]:
df = df[df["Age"] >= 0]
df["Age"].min() # Verify Invalid age removel

0

### Convert date columns

In [69]:
df["ScheduledDay"] = pd.to_datetime(df["ScheduledDay"])
df["AppointmentDay"] = pd.to_datetime(df["AppointmentDay"])

In [70]:
df[["ScheduledDay", "AppointmentDay"]].dtypes # Verify the change

ScheduledDay      datetime64[us, UTC]
AppointmentDay    datetime64[us, UTC]
dtype: object

### Fix invalid dates

In [71]:
invalid_dates = df[
    df["ScheduledDay"].dt.date > df["AppointmentDay"].dt.date
]
print("Invalid date records:", len(invalid_dates))

Invalid date records: 5


In [72]:
invalid_dates["Date_Difference"] = (
    invalid_dates["AppointmentDay"].dt.date -
    invalid_dates["ScheduledDay"].dt.date
)
invalid_dates[[
    "ScheduledDay",
    "AppointmentDay",
    "Date_Difference"
]]

,ScheduledDay,AppointmentDay,Date_Difference
27033,2016-05-10 10:51:53+00:00,2016-05-09 00:00:00+00:00,"-1 day, 0:00:00"
55226,2016-05-18 14:50:41+00:00,2016-05-17 00:00:00+00:00,"-1 day, 0:00:00"
64175,2016-05-05 13:43:58+00:00,2016-05-04 00:00:00+00:00,"-1 day, 0:00:00"
71533,2016-05-11 13:49:20+00:00,2016-05-05 00:00:00+00:00,"-6 days, 0:00:00"
72362,2016-05-04 06:50:57+00:00,2016-05-03 00:00:00+00:00,"-1 day, 0:00:00"


In [73]:
df = df[df["ScheduledDay"].dt.date <= df["AppointmentDay"].dt.date] # Remove dates
print("Rows after removing invalid dates:", len(df)) # Verify

Rows after removing invalid dates: 110521


### Clean columun names

In [74]:
# Correct spelling
df.rename(columns={
    "Hipertension": "Hypertension",
    "Handcap": "Handicap",
    "No-show": "No_Show"
}, inplace=True)

In [75]:
# Make column names snake_case 
df.columns = [
    'patient_id',
    'appointment_id',
    'gender',
    'scheduled_day',
    'appointment_day',
    'age',
    'neighbourhood',
    'scholarship',
    'hypertension',
    'diabetes',
    'alcoholism',
    'handicap',
    'sms_received',
    'no_show'
]

In [76]:
df.columns

Index(['patient_id', 'appointment_id', 'gender', 'scheduled_day',
       'appointment_day', 'age', 'neighbourhood', 'scholarship',
       'hypertension', 'diabetes', 'alcoholism', 'handicap', 'sms_received',
       'no_show'],
      dtype='str')

### Verify cleaning again

In [77]:
print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nMinimum age:", df["age"].min())

print("\nMaximum age:", df["age"].max())

print("\nData types:")
print(df.dtypes)

Dataset shape: (110521, 14)

Missing values:
patient_id         0
appointment_id     0
gender             0
scheduled_day      0
appointment_day    0
age                0
neighbourhood      0
scholarship        0
hypertension       0
diabetes           0
alcoholism         0
handicap           0
sms_received       0
no_show            0
dtype: int64

Duplicate rows: 0

Minimum age: 0

Maximum age: 115

Data types:
patient_id                     float64
appointment_id                   int64
gender                             str
scheduled_day      datetime64[us, UTC]
appointment_day    datetime64[us, UTC]
age                              int64
neighbourhood                      str
scholarship                      int64
hypertension                     int64
diabetes                         int64
alcoholism                       int64
handicap                         int64
sms_received                     int64
no_show                            str
dtype: object


In [78]:
invalid_dates = df[
    df["scheduled_day"].dt.date > df["appointment_day"].dt.date
]
print("Remaining invalid date records:", len(invalid_dates))

Remaining invalid date records: 0


In [79]:
df.to_csv("Medical_Appointment_No_Shows_Cleaned.csv", index=False)
print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
